## Automatic Music Transcription

Automatic music transcription (AMT) estimates note information from an audio signal and converts it into MIDI.

This notebook demonstrates three transcription models:

- **TransKun**: an automatic transcription model specialized for piano music https://github.com/yujia-yan/transkun
- **STRAdi**: an offline transcription model specialized for solo violin https://github.com/seingreen/STRAdi
- **Basic Pitch**: a lightweight polyphonic transcription model for various instruments https://github.com/spotify/basic-pitch

## 0. Environment Setup

Install the transcription and MIDI synthesis dependencies. Run this cell once for each new Colab runtime.

In [ ]:
%cd /content
import os

if not os.path.isdir("/content/ksmpc2026/.git"):
    !git clone https://github.com/laurenceyoon/ksmpc2026.git /content/ksmpc2026
else:
    !git -C /content/ksmpc2026 pull --ff-only

%cd /content/ksmpc2026/notebooks

In [ ]:
# Install the required Linux programs and Python packages.
import subprocess
import sys

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "fluidsynth", "fluid-soundfont-gm"],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transkun==2.0.1", "pretty_midi", "pyfluidsynth",
        "mir_eval", "resampy<0.4.3", "onnxruntime",
        "torchlibrosa", "mido", "soundfile",
    ],
    check=True,
)

# Use the ONNX model to avoid TensorFlow compatibility issues in Colab.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "basic-pitch==0.4.0"],
    check=True,
)

print("Package installation complete.")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
RESOURCE_DIR = PROJECT_ROOT / "resources"
CHOPIN_AUDIOS = {
    "p09": RESOURCE_DIR / "Chopin_op10_no3_p09_short.wav",
    "p15": RESOURCE_DIR / "Chopin_op10_no3_p15_short.wav",
}
CHOPIN_SCORE = RESOURCE_DIR / "Chopin_op10_no3_p15_short.png"
BEETHOVEN_AUDIO = RESOURCE_DIR / "Beethoven_strqrt.wav"
BEETHOVEN_SCORE = RESOURCE_DIR / "Beethovon_String_Quartet_No.1_Op_18.png"
STRADI_AUDIO = RESOURCE_DIR / "Paganini_Caprice_No.24_Op.1_BomsoriKim.wav"
STRADI_SCORE = RESOURCE_DIR / "Paganini_Caprice_No.24_Op.1.png"
OUTPUT_DIR = PROJECT_ROOT / "results" / "automatic_music_transcription"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import the libraries used throughout the notebook.
import numpy as np
import matplotlib.pyplot as plt
import pretty_midi
import IPython.display as ipd

print("Setup complete.")

# 1. Piano Transcription with TransKun

TransKun is an automatic transcription model trained for piano music. This section uses two short excerpts from Chopin's Étude Op. 10, No. 3.

While listening, consider the following questions.

- How many notes overlap at a given time?
- How does sustain pedal affect note-duration estimation?
- How are performance dynamics represented by MIDI velocity?

In [ ]:
# Display the score, then listen to the two excerpts used by TransKun.
display(ipd.Image(filename=str(CHOPIN_SCORE), width=900))

for excerpt, audio_path in CHOPIN_AUDIOS.items():
    print(f"Chopin Op. 10 No. 3 ({excerpt}) — {audio_path.name}")
    display(ipd.Audio(filename=str(audio_path)))

### Running TransKun

A GPU is used automatically when available. The first run may take longer while the model weights are prepared.

In [ ]:
import torch

CHOPIN_MIDIS = {
    excerpt: OUTPUT_DIR / f"Chopin_op10_no3_{excerpt}_short_transkun.mid"
    for excerpt in CHOPIN_AUDIOS
}
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

for excerpt, audio_path in CHOPIN_AUDIOS.items():
    midi_path = CHOPIN_MIDIS[excerpt]
    print(f"Transcribing: {audio_path.name}")
    subprocess.run(
        ["transkun", "--device", device, str(audio_path), str(midi_path)],
        check=True,
    )
    print(f"Saved transcription: {midi_path}")

### Visualizing the TransKun Piano Rolls

The horizontal axis represents time and the vertical axis represents MIDI pitch. Color intensity represents velocity.

In [ ]:
pm_chopin = {
    excerpt: pretty_midi.PrettyMIDI(str(midi_path))
    for excerpt, midi_path in CHOPIN_MIDIS.items()
}

fig, axes = plt.subplots(len(pm_chopin), 1, figsize=(12, 7), constrained_layout=True)
for ax, (excerpt, midi_data) in zip(axes, pm_chopin.items()):
    piano_roll = midi_data.get_piano_roll(fs=100)
    image = ax.imshow(
        piano_roll,
        origin="lower",
        aspect="auto",
        cmap="magma",
        extent=[0, midi_data.get_end_time(), 0, 127],
    )
    ax.set_title(f"TransKun: Chopin Op. 10 No. 3 ({excerpt})")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("MIDI pitch")
    ax.set_ylim(20, 110)
    fig.colorbar(image, ax=ax, label="Velocity")
plt.show()

### Listening to the Generated MIDI

The transcription is synthesized with FluidSynth. Compare it with the original recording and listen for missing notes, extra notes, and duration errors.

In [ ]:
SOUNDFONT = Path("/usr/share/sounds/sf2/FluidR3_GM.sf2")
if not SOUNDFONT.exists():
    soundfonts = list(Path("/usr/share/sounds").rglob("*.sf2"))
    if not soundfonts:
        raise FileNotFoundError("No SoundFont file was found.")
    SOUNDFONT = soundfonts[0]

for excerpt, midi_data in pm_chopin.items():
    print(f"TransKun synthesis ({excerpt})")
    chopin_synth = midi_data.fluidsynth(fs=44100, sf2_path=str(SOUNDFONT))
    display(ipd.Audio(chopin_synth, rate=44100))

# 2. Solo-Violin Transcription with STRAdi

[STRAdi](https://github.com/seingreen/STRAdi) converts solo-violin audio into note events. This section uses the non-causal offline model on Bomsori Kim's performance of Paganini's Caprice No. 24, Op. 1.

Unlike the causal online model, the offline model can use both past and future context from the complete recording.

### Preparing STRAdi

The source repository and offline checkpoint are downloaded once and cached inside the project directory.

In [ ]:
import urllib.request

STRADI_DIR = PROJECT_ROOT / ".cache" / "STRAdi"
STRADI_CHECKPOINT = STRADI_DIR / "checkpoints" / "violin_transcription_offline.pth"
STRADI_CHECKPOINT_URL = (
    "https://github.com/seingreen/STRAdi/releases/download/v1.0.0/"
    "violin_transcription_offline.pth"
)

if not (STRADI_DIR / "transcribe.py").is_file():
    STRADI_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/seingreen/STRAdi.git", str(STRADI_DIR)],
        check=True,
    )

# PyTorch 2.6+ defaults to weights_only=True, but this trusted release
# checkpoint contains NumPy metadata in addition to model weights.
checkpoint_loader = STRADI_DIR / "violin_transcription" / "checkpoint.py"
loader_source = checkpoint_loader.read_text()
old_load_call = "torch.load(str(checkpoint_path), map_location=device)"
new_load_call = (
    "torch.load(str(checkpoint_path), map_location=device, weights_only=False)"
)
if old_load_call in loader_source:
    checkpoint_loader.write_text(loader_source.replace(old_load_call, new_load_call))

if not STRADI_CHECKPOINT.is_file():
    STRADI_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading the STRAdi offline checkpoint...")
    urllib.request.urlretrieve(STRADI_CHECKPOINT_URL, STRADI_CHECKPOINT)

display(ipd.Image(filename=str(STRADI_SCORE), width=900))
print(f"STRAdi input: {STRADI_AUDIO}")
display(ipd.Audio(filename=str(STRADI_AUDIO)))

### TransKun on the Solo-Violin Recording

TransKun is trained for piano, so this is an out-of-domain test rather than its intended use.

In [ ]:
PAGANINI_TRANSKUN_MIDI = OUTPUT_DIR / f"{STRADI_AUDIO.stem}_transkun.mid"
subprocess.run(
    ["transkun", "--device", device, str(STRADI_AUDIO), str(PAGANINI_TRANSKUN_MIDI)],
    check=True,
)

pm_paganini_transkun = pretty_midi.PrettyMIDI(str(PAGANINI_TRANSKUN_MIDI))
paganini_transkun_roll = pm_paganini_transkun.get_piano_roll(fs=100)

plt.figure(figsize=(12, 4))
plt.imshow(
    paganini_transkun_roll,
    origin="lower",
    aspect="auto",
    cmap="magma",
    extent=[0, pm_paganini_transkun.get_end_time(), 0, 127],
)
plt.title("TransKun transcription: Paganini Caprice No. 24")
plt.xlabel("Time (seconds)")
plt.ylabel("MIDI pitch")
plt.ylim(50, 115)
plt.colorbar(label="Velocity")
plt.show()

paganini_transkun_synth = pm_paganini_transkun.fluidsynth(
    fs=44100, sf2_path=str(SOUNDFONT)
)
ipd.Audio(paganini_transkun_synth, rate=44100)

### Running the STRAdi Offline Model

The transcription is written as both MIDI and CSV note events with onset, offset, and MIDI pitch.

In [ ]:
STRADI_OUTPUT_DIR = OUTPUT_DIR / "stradi_offline"
STRADI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stradi_process = subprocess.run(
    [
        sys.executable, str(STRADI_DIR / "transcribe.py"), str(STRADI_AUDIO),
        "--checkpoint", str(STRADI_CHECKPOINT),
        "--model", "offline",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--output-dir", str(STRADI_OUTPUT_DIR),
    ],
    cwd=STRADI_DIR,
    capture_output=True,
    text=True,
)
print(stradi_process.stdout)
if stradi_process.returncode != 0:
    print(stradi_process.stderr)
    stradi_process.check_returncode()

STRADI_MIDI = STRADI_OUTPUT_DIR / f"{STRADI_AUDIO.stem}.mid"
STRADI_CSV = STRADI_OUTPUT_DIR / f"{STRADI_AUDIO.stem}.csv"
print(f"Saved MIDI: {STRADI_MIDI}")
print(f"Saved note events: {STRADI_CSV}")

### STRAdi Offline Transcription

The piano roll shows the notes detected by the offline solo-violin model.

In [ ]:
pm_stradi = pretty_midi.PrettyMIDI(str(STRADI_MIDI))
stradi_roll = pm_stradi.get_piano_roll(fs=100)

plt.figure(figsize=(12, 4))
plt.imshow(
    stradi_roll,
    origin="lower",
    aspect="auto",
    cmap="magma",
    extent=[0, pm_stradi.get_end_time(), 0, 127],
)
plt.title("STRAdi offline transcription: Paganini Caprice No. 24")
plt.xlabel("Time (seconds)")
plt.ylabel("MIDI pitch")
plt.ylim(50, 115)
plt.colorbar(label="Velocity")
plt.show()

stradi_synth = pm_stradi.fluidsynth(fs=44100, sf2_path=str(SOUNDFONT))
ipd.Audio(stradi_synth, rate=44100)

# 3. Multi-Instrument Transcription with Basic Pitch

Basic Pitch is a lightweight automatic transcription model released by Spotify. This section uses a Beethoven string quartet recording.

String transcription presents several challenges compared with piano transcription.

- Note onsets may be less distinct.
- Vibrato and portamento produce continuous pitch variation.
- Multiple instruments may play simultaneously in similar pitch ranges.

In [ ]:
# Display the Beethoven score and listen to the input audio.
display(ipd.Image(filename=str(BEETHOVEN_SCORE), width=1000))
display(ipd.Audio(filename=str(BEETHOVEN_AUDIO)))

### Running Basic Pitch

This notebook uses the ONNX model for Colab compatibility. The model returns its frame-level output, a MIDI object, and a list of note events.

In [ ]:
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

print(f"Model: {ICASSP_2022_MODEL_PATH}")
model_output, midi_data, note_events = predict(
    str(BEETHOVEN_AUDIO),
    model_or_model_path=ICASSP_2022_MODEL_PATH,
)

BEETHOVEN_MIDI = OUTPUT_DIR / "beethoven_basic_pitch.mid"
midi_data.write(str(BEETHOVEN_MIDI))
print(f"Detected notes: {len(note_events)}")
print(f"Saved MIDI: {BEETHOVEN_MIDI}")

### Estimated Pitch Distribution

The plot shows pitch probabilities over time. Vibrato appears as vertical oscillation in the activation pattern.

In [ ]:
plt.figure(figsize=(12, 5))
plt.imshow(
    model_output["contour"].T,
    aspect="auto",
    origin="lower",
    cmap="magma",
)
plt.title("Basic Pitch: pitch contour")
plt.xlabel("Frame")
plt.ylabel("Pitch bin")
plt.colorbar(label="Confidence")
plt.show()

### MIDI Piano Roll

This piano roll shows the note onsets and offsets inferred from the continuous pitch probabilities.

In [ ]:
beethoven_roll = midi_data.get_piano_roll(fs=100)

plt.figure(figsize=(12, 4))
plt.imshow(
    beethoven_roll,
    origin="lower",
    aspect="auto",
    cmap="viridis",
    extent=[0, midi_data.get_end_time(), 0, 127],
)
plt.title("Basic Pitch transcription: Beethoven String Quartet")
plt.xlabel("Time (seconds)")
plt.ylabel("MIDI pitch")
plt.ylim(20, 110)
plt.colorbar(label="Velocity")
plt.show()

### Notes and Pitch Bends

Blue lines represent MIDI notes. Red lines show the estimated pitch variation within each note.

In [ ]:
plt.figure(figsize=(12, 4))

for start, end, pitch, velocity, pitch_bend in note_events:
    plt.plot([start, end], [pitch, pitch], color="royalblue", linewidth=2)
    if pitch_bend:
        times = np.linspace(start, end, len(pitch_bend))
        plt.plot(
            times,
            pitch + np.asarray(pitch_bend),
            color="crimson",
            alpha=0.55,
        )

plt.title("MIDI notes (blue) and pitch bends (red)")
plt.xlabel("Time (seconds)")
plt.ylabel("MIDI pitch")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# List and optionally download the generated files.
from google.colab import files

print("Generated files:")
for path in sorted(OUTPUT_DIR.rglob("*.mid")):
    print(" -", path.relative_to(OUTPUT_DIR))

# Uncomment these lines to download the files from Colab.
# for path in CHOPIN_MIDIS.values():
#     files.download(str(path))
# files.download(str(BEETHOVEN_MIDI))
# files.download(str(STRADI_MIDI))
# files.download(str(STRADI_CSV))